In [ ]:
# Step 01 — Global configuration

from pathlib import Path

REPO_URL = "https://github.com/YutingLi0606/Idempotent-Continual-Learning.git"
ROOT = Path("/kaggle/working")
REPO = ROOT / "Idempotent-Continual-Learning"
OUT = ROOT / "ider_reproduction"
DATA_ROOT = ROOT / "data"

OUT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    "cifar10_b200":   dict(dataset="seq-cifar10",  buffer=200,  tasks=5),
    "cifar10_b500":   dict(dataset="seq-cifar10",  buffer=500,  tasks=5),
    "cifar100_b500":  dict(dataset="seq-cifar100", buffer=500,  tasks=10),
    "cifar100_b2000": dict(dataset="seq-cifar100", buffer=2000, tasks=10),
    "tiny_b500":      dict(dataset="seq-tinyimg",  buffer=500,  tasks=10),
    "tiny_b4000":     dict(dataset="seq-tinyimg",  buffer=4000, tasks=10),
}

ACTIVE_EXPERIMENTS = list(EXPERIMENTS)
SEEDS = [0, 1, 2, 3, 4]
CLASS_BALANCE = True
DEVICE = "cuda:0"

print("Experiments:", ACTIVE_EXPERIMENTS)
print("Seeds:", SEEDS)
print("Total runs:", len(ACTIVE_EXPERIMENTS) * len(SEEDS))


In [ ]:
# Step 02 — Python / OS information

import sys, platform, os
print("Python:", sys.version)
print("OS:", platform.platform())
print("Working directory:", ROOT)
print("CPU count:", os.cpu_count())


In [ ]:
# Step 03 — GPU / CUDA information

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle before continuing.")

print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("GPU memory (GB):", round(props.total_memory / 1024**3, 2))


In [ ]:
# Step 04 — torchvision information

import torchvision
print("torchvision:", torchvision.__version__)


In [ ]:
# Step 05 — Reproducibility environment flags
# The authors' own seed handling remains authoritative inside main.py.
# These environment variables reduce avoidable nondeterminism without changing IDER's objective.

import os
os.environ["PYTHONHASHSEED"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
print("Environment flags set.")


In [ ]:
# Step 06 — Install auxiliary dependencies commonly absent from Kaggle.
# Do not downgrade Kaggle PyTorch automatically.

import importlib.util, subprocess, sys

REQ = {
    "mlflow": "mlflow",
    "setproctitle": "setproctitle",
    "onedrivedownloader": "onedrivedownloader",
    "torch_optimizer": "torch-optimizer",
    "randaugment": "randaugment",
    "easydict": "easydict",
    "timm": "timm",
}

missing = [pip for module, pip in REQ.items() if importlib.util.find_spec(module) is None]
print("Missing:", missing)

if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


In [ ]:
# Step 07 — Clone official repository

import subprocess, shutil

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
else:
    print("Using existing repository:", REPO)

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()

print("Commit:", commit)
(OUT / "repo_commit.txt").write_text(commit + "\n")


In [ ]:
# Step 08 — Repository tree

for p in sorted(REPO.rglob("*")):
    if p.is_file() and ".git" not in p.parts:
        rel = p.relative_to(REPO)
        if len(rel.parts) <= 2:
            print(rel)


In [ ]:
# Step 09 — Required-file integrity check

required = [
    "main.py",
    "models/ider.py",
    "models/er.py",
    "utils/buffer.py",
    "utils/best_args.py",
    "utils/training.py",
    "backbone/ResNet18_id2.py",
    "datasets/seq_cifar10.py",
    "datasets/seq_cifar100.py",
    "datasets/seq_tinyimagenet.py",
    "run_para_cifar10.sh",
    "run_para_cifar100.sh",
    "run_para_tinyimg.sh",
]

missing_files = [x for x in required if not (REPO / x).exists()]
print("Missing required files:", missing_files)
if missing_files:
    raise FileNotFoundError(missing_files)


In [ ]:
# Step 10 — Hash important source files for provenance

import hashlib, pandas as pd

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

hash_rows = [{"file": f, "sha256": sha256(REPO/f)} for f in required]
hash_df = pd.DataFrame(hash_rows)
display(hash_df)
hash_df.to_csv(OUT/"source_hashes.csv", index=False)


In [ ]:
# Step 11 — Utility for source inspection

def show_source(rel, start=1, end=None):
    path = REPO / rel
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    end = len(lines) if end is None else min(end, len(lines))
    print(f"\n### {rel} | lines {start}-{end}\n")
    for i in range(start-1, end):
        print(f"{i+1:4d}: {lines[i]}")


In [ ]:
# Step 12 — Inspect IDER model implementation
show_source("models/ider.py")


In [ ]:
# Step 13 — Inspect replay buffer implementation
show_source("utils/buffer.py")


In [ ]:
# Step 14 — Inspect modified ResNet-18
show_source("backbone/ResNet18_id2.py")


In [ ]:
# Step 15 — Inspect training loop
show_source("utils/training.py")


In [ ]:
# Step 16 — Inspect best hyperparameters
show_source("utils/best_args.py")


In [ ]:
# Step 17 — Inspect CIFAR-10 dataset/task pipeline
show_source("datasets/seq_cifar10.py")


In [ ]:
# Step 18 — Inspect CIFAR-100 dataset/task pipeline
show_source("datasets/seq_cifar100.py")


In [ ]:
# Step 19 — TinyImageNet pipeline audit (concise)

from pathlib import Path
import ast

tiny_path = REPO / "datasets/seq_tinyimagenet.py"
tiny_src = tiny_path.read_text(encoding="utf-8", errors="replace")
tiny_tree = ast.parse(tiny_src)

# Find the continual-learning dataset class.
tiny_cls = next(
    node for node in tiny_tree.body
    if isinstance(node, ast.ClassDef) and node.name == "SequentialTinyImagenet"
)

# Evaluate simple class constants in source order.
env = {}

def _eval_simple(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.Name) and node.id in env:
        return env[node.id]
    if isinstance(node, ast.BinOp) and isinstance(node.op, ast.FloorDiv):
        return _eval_simple(node.left) // _eval_simple(node.right)
    return None

for stmt in tiny_cls.body:
    if isinstance(stmt, ast.Assign) and len(stmt.targets) == 1 and isinstance(stmt.targets[0], ast.Name):
        value = _eval_simple(stmt.value)
        if value is not None:
            env[stmt.targets[0].id] = value

# Read simple constant-return methods.
def _constant_return(method_name):
    fn = next(
        (n for n in tiny_cls.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == method_name),
        None
    )
    if fn is None:
        return None
    for stmt in ast.walk(fn):
        if isinstance(stmt, ast.Return) and isinstance(stmt.value, ast.Constant):
            return stmt.value.value
    return None

default_epochs = _constant_return("get_epochs")
batch_size = _constant_return("get_batch_size")

print("TinyImageNet source:", tiny_path)
print("-" * 68)
print(f"Dataset name        : {env.get('NAME')}")
print(f"Setting             : {env.get('SETTING')}")
print(f"Total classes       : {env.get('N_CLASSES')}")
print(f"Tasks               : {env.get('N_TASKS')}")
print(f"Classes / task      : {env.get('N_CLASSES_PER_TASK')}")
print(f"Dataset default epoch: {default_epochs}")
print(f"Default batch size  : {batch_size}")
print(f"IDER backbone hook  : {'resnet18_id2' if 'resnet18_id2' in tiny_src else 'NOT FOUND'}")
print(f"Replay transform    : {'get_transform' if 'def get_transform' in tiny_src else 'NOT FOUND'}")
print(f"Auto-download code  : {'present' if 'onedrivedownloader' in tiny_src else 'not found'}")

# Show only the scheduler definitions instead of dumping the full ~180-line file.
scheduler_lines = [
    f"{i:03d}: {line.strip()}"
    for i, line in enumerate(tiny_src.splitlines(), start=1)
    if "MultiStepLR" in line or "args.n_epochs==50" in line
]

print("\nScheduler logic:")
for line in scheduler_lines:
    print(" ", line)

# Sanity checks for the official Split TinyImageNet setup.
assert env.get("NAME") == "seq-tinyimg"
assert env.get("SETTING") == "class-il"
assert env.get("N_TASKS") == 10
assert env.get("N_CLASSES") == 200
assert env.get("N_CLASSES_PER_TASK") == 20
assert batch_size == 32

print("\n[OK] TinyImageNet continual-learning pipeline passed structural checks.")


In [ ]:
# Step 20 — Inspect authors' CIFAR-10 run script
show_source("run_para_cifar10.sh")


In [ ]:
# Step 21 — Inspect authors' CIFAR-100 run script
show_source("run_para_cifar100.sh")


In [ ]:
# Step 22 — Inspect authors' TinyImageNet run script
show_source("run_para_tinyimg.sh")


In [ ]:
# Step 23 — Import official best_args.py

import importlib.util

spec = importlib.util.spec_from_file_location("ider_best_args", REPO/"utils/best_args.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
BEST_ARGS = mod.best_args

print("Loaded best_args from:", REPO/"utils/best_args.py")


In [ ]:
# Step 24 — Print exact IDER hyperparameters for every requested setting

import pprint

for name in ACTIVE_EXPERIMENTS:
    e = EXPERIMENTS[name]
    print("\n", "="*80)
    print(name)
    pprint.pp(BEST_ARGS[e["dataset"]]["ider"][e["buffer"]])


In [ ]:
# Step 25 — Build a compact hyperparameter audit table

rows = []
for name in ACTIVE_EXPERIMENTS:
    e = EXPERIMENTS[name]
    hp = dict(BEST_ARGS[e["dataset"]]["ider"][e["buffer"]])
    rows.append({
        "experiment": name,
        "dataset": e["dataset"],
        "buffer": e["buffer"],
        "tasks": e["tasks"],
        **hp,
    })

hp_df = pd.DataFrame(rows)
display(hp_df)
hp_df.to_csv(OUT/"official_best_args.csv", index=False)


In [ ]:
# Step 26 — Confirm class-balance mode used by this reproduction

print("CLASS_BALANCE =", CLASS_BALANCE)
assert CLASS_BALANCE is True


In [ ]:
# Step 27 — Locate reservoir/class-balance logic in buffer.py

text = (REPO/"utils/buffer.py").read_text(encoding="utf-8", errors="replace").splitlines()
keywords = ["reservoir", "class_balance", "num_seen_examples", "examples", "labels"]
for kw in keywords:
    print(f"\n--- matches for {kw!r} ---")
    for i, line in enumerate(text, 1):
        if kw.lower() in line.lower():
            print(f"{i:4d}: {line}")


In [ ]:
# Step 28 — Locate buffer calls inside IDER

text = (REPO/"models/ider.py").read_text(encoding="utf-8", errors="replace").splitlines()
for i, line in enumerate(text, 1):
    if "buffer" in line.lower():
        print(f"{i:4d}: {line}")


In [ ]:
# Step 29 — Locate second-input / IDER-specific architecture operations

text = (REPO/"backbone/ResNet18_id2.py").read_text(encoding="utf-8", errors="replace").splitlines()
for i, line in enumerate(text, 1):
    low = line.lower()
    if any(k in low for k in ["linear", "leaky", "layer2", "layer3", "forward", "num_classes"]):
        print(f"{i:4d}: {line}")


In [ ]:
# Step 30 — Compile-check repository Python files

import py_compile

failed = []
for p in REPO.rglob("*.py"):
    if ".git" in p.parts:
        continue
    try:
        py_compile.compile(str(p), doraise=True)
    except Exception as exc:
        failed.append((str(p.relative_to(REPO)), repr(exc)))

print("Compile failures:", len(failed))
for x in failed[:30]:
    print(x)


In [ ]:
# Step 31 — Back up files before compatibility patch

import shutil

backup = OUT/"source_backup"
backup.mkdir(exist_ok=True)

for rel in ["datasets/seq_cifar100.py", "datasets/seq_tinyimagenet.py", "utils/training.py"]:
    src = REPO/rel
    dst = backup/rel.replace("/", "__")
    if not dst.exists():
        shutil.copy2(src, dst)

print("Backups:", list(backup.iterdir()))


In [ ]:
# Step 32 — Remove deprecated scheduler verbose=False only if present

for rel in ["datasets/seq_cifar100.py", "datasets/seq_tinyimagenet.py"]:
    p = REPO/rel
    s = p.read_text(encoding="utf-8")
    s2 = s.replace(", verbose=False)", ")")
    if s2 != s:
        p.write_text(s2, encoding="utf-8")
        print("Patched:", rel)
    else:
        print("No patch needed:", rel)


In [ ]:
# Step 33 — Measurement-only patch: export task-end CIL/TIL snapshots to JSON.
# It does not modify model parameters, losses, optimizer, scheduler, data, or buffer.

p = REPO/"utils/training.py"
s = p.read_text(encoding="utf-8")
marker = "IDER_KAGGLE_METRIC_EXPORT_V2"

if marker not in s:
    anchor1 = "    results, results_mask_classes = [], []"
    if anchor1 not in s:
        raise RuntimeError("training.py anchor 1 not found; inspect repository before continuing.")
    repl1 = anchor1 + """
    # IDER_KAGGLE_METRIC_EXPORT_V2
    _ider_cil_snapshots = []
    _ider_til_snapshots = []
"""
    s = s.replace(anchor1, repl1, 1)

    anchor2 = """        results.append(accs[0])
        results_mask_classes.append(accs[1])"""
    if anchor2 not in s:
        raise RuntimeError("training.py anchor 2 not found; inspect repository before continuing.")
    repl2 = anchor2 + """
        # IDER_KAGGLE_METRIC_EXPORT_V2: measurement only
        _ider_cil_snapshots.append([float(v) for v in accs[0]])
        _ider_til_snapshots.append([float(v) for v in accs[1]])
        _export = os.environ.get("IDER_KAGGLE_EXPORT")
        if _export:
            import json as _json
            with open(_export, "w", encoding="utf-8") as _f:
                _json.dump({
                    "task_completed": int(t + 1),
                    "cil_snapshots": _ider_cil_snapshots,
                    "til_snapshots": _ider_til_snapshots,
                }, _f, indent=2)
"""
    s = s.replace(anchor2, repl2, 1)
    p.write_text(s, encoding="utf-8")
    print("Metric export patch applied.")
else:
    print("Metric export patch already present.")


In [ ]:
# Step 34 — Re-compile patched files

for rel in ["datasets/seq_cifar100.py", "datasets/seq_tinyimagenet.py", "utils/training.py"]:
    py_compile.compile(str(REPO/rel), doraise=True)
    print("OK:", rel)


In [ ]:
# Step 35 — CLI smoke test

import subprocess, sys

r = subprocess.run(
    [sys.executable, "main.py", "--help"],
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(r.stdout[:8000])
if r.returncode != 0:
    raise RuntimeError("Official repository CLI failed before training.")


In [ ]:
# Step 36 — Create the 30-run manifest

manifest = []
for exp_name in ACTIVE_EXPERIMENTS:
    e = EXPERIMENTS[exp_name]
    for seed in SEEDS:
        manifest.append({
            "experiment": exp_name,
            "dataset": e["dataset"],
            "buffer": e["buffer"],
            "tasks": e["tasks"],
            "seed": seed,
        })

manifest_df = pd.DataFrame(manifest)
display(manifest_df)
manifest_df.to_csv(OUT/"run_manifest.csv", index=False)
print("Total:", len(manifest_df))


In [ ]:
# Step 37 — Disk-space check

import shutil
usage = shutil.disk_usage(ROOT)
print("Total GB:", round(usage.total/1024**3, 2))
print("Used  GB:", round(usage.used/1024**3, 2))
print("Free  GB:", round(usage.free/1024**3, 2))


In [ ]:
# Step 38 — Metric helper

import numpy as np

def metrics_from_snapshots(payload):
    cil = payload["cil_snapshots"]
    til = payload["til_snapshots"]
    if not cil:
        raise ValueError("No snapshots.")

    cil_faa = float(np.mean(cil[-1]))
    til_faa = float(np.mean(til[-1]))

    forgetting = []
    T = len(cil)
    for task_i in range(T-1):
        history = [cil[j][task_i] for j in range(task_i, T) if len(cil[j]) > task_i]
        forgetting.append(max(history) - cil[-1][task_i])

    return {
        "CIL_FAA": cil_faa,
        "TIL_FAA": til_faa,
        "Final_Forgetting": float(np.mean(forgetting)) if forgetting else 0.0,
    }


In [ ]:
# Step 39 — Define one-run executor

import time, json, subprocess, sys, os

def run_one(exp_name, seed, force=False):
    e = EXPERIMENTS[exp_name]
    run_dir = OUT/exp_name/f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    summary_file = run_dir/"summary.json"
    snapshots_file = run_dir/"task_snapshots.json"
    log_file = run_dir/"train.log"

    if summary_file.exists() and not force:
        print("SKIP completed:", exp_name, seed)
        return json.loads(summary_file.read_text())

    cmd = [
        sys.executable, "main.py",
        "--model=ider",
        "--load_best_args",
        "--savecheckpoint=True",
        f"--class_balance={str(CLASS_BALANCE)}",
        f"--dataset={e['dataset']}",
        f"--device={DEVICE}",
        f"--seed={seed}",
        f"--n_tasks={e['tasks']}",
        f"--buffer_size={e['buffer']}",
        f"--run_name=kaggle_{exp_name}_seed{seed}",
        f"--experiment_name=kaggle_reproduction/{exp_name}",
        "--non_verbose",
        "--disable_log",
    ]

    env = os.environ.copy()
    env["IDER_KAGGLE_EXPORT"] = str(snapshots_file)
    env["PYTHONUNBUFFERED"] = "1"

    print("\n" + "="*100)
    print("EXPERIMENT:", exp_name, "| SEED:", seed)
    print(" ".join(cmd))
    print("="*100)

    start = time.time()
    with log_file.open("w", encoding="utf-8", buffering=1) as fout:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            fout.write(line)
        rc = proc.wait()

    elapsed = (time.time() - start)/60

    if rc != 0:
        raise RuntimeError(f"Run failed: {exp_name}, seed={seed}. See {log_file}")
    if not snapshots_file.exists():
        raise RuntimeError(f"No snapshot JSON: {snapshots_file}")

    payload = json.loads(snapshots_file.read_text())
    m = metrics_from_snapshots(payload)

    summary = {
        "experiment": exp_name,
        "dataset": e["dataset"],
        "buffer": e["buffer"],
        "tasks": e["tasks"],
        "seed": seed,
        "minutes": elapsed,
        **m,
    }
    summary_file.write_text(json.dumps(summary, indent=2))
    return summary


In [ ]:
# Step 40 — Single-run dry manifest check (does not train)

example = manifest_df.iloc[0].to_dict()
print(example)
print("Ready to train:", len(manifest_df), "runs")


In [ ]:
# Step 41 — RUN ALL 30 EXPERIMENTS

all_results = []

for exp_name in ACTIVE_EXPERIMENTS:
    for seed in SEEDS:
        result = run_one(exp_name, seed, force=False)
        all_results.append(result)

print("\nFinished/loaded:", len(all_results), "runs")


In [ ]:
# Step 42 — Reload all completed summaries from disk

summary_files = sorted(OUT.glob("*/seed_*/summary.json"))
records = [json.loads(p.read_text()) for p in summary_files]

results = pd.DataFrame(records)
if results.empty:
    raise RuntimeError("No completed runs found.")

results = results.sort_values(["experiment", "seed"]).reset_index(drop=True)
display(results)
results.to_csv(OUT/"per_seed_results.csv", index=False)


In [ ]:
# Step 43 — Verify seed completeness

completeness = (
    results.groupby("experiment")["seed"]
    .agg(["count", lambda s: sorted(s.tolist())])
    .reset_index()
)
completeness.columns = ["experiment", "n_runs", "seeds"]
display(completeness)


In [ ]:
# Step 44 — Five-seed mean ± std

agg = results.groupby("experiment").agg(
    n_runs=("seed", "count"),
    CIL_mean=("CIL_FAA", "mean"),
    CIL_std=("CIL_FAA", "std"),
    TIL_mean=("TIL_FAA", "mean"),
    TIL_std=("TIL_FAA", "std"),
    FF_mean=("Final_Forgetting", "mean"),
    FF_std=("Final_Forgetting", "std"),
    minutes_mean=("minutes", "mean"),
).reset_index()

display(agg)
agg.to_csv(OUT/"five_seed_aggregate.csv", index=False)


In [ ]:
# Step 45 — Human-readable mean ± std table

pretty = agg.copy()
pretty["CIL FAA"] = pretty.apply(lambda r: f"{r.CIL_mean:.2f} ± {r.CIL_std:.2f}", axis=1)
pretty["TIL FAA (extra diagnostic)"] = pretty.apply(
    lambda r: f"{r.TIL_mean:.2f} ± {r.TIL_std:.2f}", axis=1
)
pretty["Final Forgetting"] = pretty.apply(
    lambda r: f"{r.FF_mean:.2f} ± {r.FF_std:.2f}", axis=1
)

display(
    pretty[
        [
            "experiment",
            "n_runs",
            "CIL FAA",
            "Final Forgetting",
            "TIL FAA (extra diagnostic)",
        ]
    ]
)

print("Note: TIL FAA is an additional diagnostic; it is not reported in the paper's main CIL tables.")


In [ ]:
# Step 46 — Paper reference values (Table 1 FAA + Table 7 FF)

# Exact ER+ID (Ours) values transcribed from the uploaded ICLR 2026 paper.
# Table 1: Final Average Accuracy (FAA) under Class-IL.
# Table 7: Final Forgetting (FF).
# TIL is intentionally NOT included because the paper's main tables do not report TIL FAA.

paper_reference = pd.DataFrame([
    ["cifar10_b200",    71.02, 1.98, 15.28, 2.41],
    ["cifar10_b500",    74.74, 0.42, 11.93, 0.49],
    ["cifar100_b500",   44.82, 0.85, 29.98, 2.52],
    ["cifar100_b2000",  56.59, 0.35, 17.46, 1.04],
    ["tiny_b500",       29.88, 1.15, 36.63, 3.37],
    ["tiny_b4000",      43.05, 1.40, 22.46, 1.86],
], columns=[
    "experiment",
    "Paper_FAA_mean",
    "Paper_FAA_std",
    "Paper_FF_mean",
    "Paper_FF_std",
])

display(paper_reference)
paper_reference.to_csv(OUT/"paper_reference_table1_table7.csv", index=False)


In [ ]:
# Step 47 — Compare Kaggle reproduction with paper-reported metrics

comparison = agg.merge(paper_reference, on="experiment", how="left")

comparison["FAA_delta_vs_paper"] = (
    comparison["CIL_mean"] - comparison["Paper_FAA_mean"]
)
comparison["FF_delta_vs_paper"] = (
    comparison["FF_mean"] - comparison["Paper_FF_mean"]
)

paper_comparison_cols = [
    "experiment",
    "n_runs",
    "CIL_mean",
    "CIL_std",
    "Paper_FAA_mean",
    "Paper_FAA_std",
    "FAA_delta_vs_paper",
    "FF_mean",
    "FF_std",
    "Paper_FF_mean",
    "Paper_FF_std",
    "FF_delta_vs_paper",
]

display(comparison[paper_comparison_cols])
comparison[paper_comparison_cols].to_csv(
    OUT/"paper_reference_comparison.csv",
    index=False,
)

print("TIL FAA is excluded from the paper comparison because the paper does not report it in the main CIL tables.")


In [ ]:
# Step 48 — CIL FAA plot

import matplotlib.pyplot as plt

plot_df = agg.sort_values("experiment")
plt.figure(figsize=(10, 5))
plt.errorbar(plot_df["experiment"], plot_df["CIL_mean"], yerr=plot_df["CIL_std"], fmt="o", capsize=4)
plt.xticks(rotation=30, ha="right")
plt.ylabel("CIL Final Average Accuracy (%)")
plt.title("IDER — 5-seed CIL FAA")
plt.tight_layout()
plt.savefig(OUT/"cil_faa.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Step 49 — TIL FAA diagnostic plot (not a paper Table 1 metric)

plt.figure(figsize=(10, 5))
plt.errorbar(
    plot_df["experiment"],
    plot_df["TIL_mean"],
    yerr=plot_df["TIL_std"],
    fmt="o",
    capsize=4,
)
plt.xticks(rotation=30, ha="right")
plt.ylabel("TIL Final Average Accuracy (%)")
plt.title("IDER — 5-seed TIL FAA (extra diagnostic)")
plt.tight_layout()
plt.savefig(OUT/"til_faa_extra_diagnostic.png", dpi=200, bbox_inches="tight")
plt.show()

print("Diagnostic only: the paper's main Class-IL result tables report FAA/FF/ECE, not TIL FAA.")


In [ ]:
# Step 50 — Final Forgetting plot

plt.figure(figsize=(10, 5))
plt.errorbar(plot_df["experiment"], plot_df["FF_mean"], yerr=plot_df["FF_std"], fmt="o", capsize=4)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Final Forgetting")
plt.title("IDER — 5-seed Final Forgetting")
plt.tight_layout()
plt.savefig(OUT/"final_forgetting.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Step 51 — Per-seed CIL scatter

plt.figure(figsize=(10, 5))
for exp_name, g in results.groupby("experiment"):
    x = [exp_name] * len(g)
    plt.scatter(x, g["CIL_FAA"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("CIL FAA (%)")
plt.title("IDER — Per-seed CIL FAA")
plt.tight_layout()
plt.savefig(OUT/"per_seed_cil.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Step 52 — Load task snapshots

snapshot_rows = []
for p in sorted(OUT.glob("*/seed_*/task_snapshots.json")):
    exp = p.parts[-3]
    seed = int(p.parts[-2].split("_")[-1])
    payload = json.loads(p.read_text())
    snapshot_rows.append((exp, seed, payload))

print("Snapshot files:", len(snapshot_rows))


In [ ]:
# Step 53 — Plot final per-task CIL accuracy for each experiment (seed mean)

for exp_name in ACTIVE_EXPERIMENTS:
    selected = [(seed, p) for exp, seed, p in snapshot_rows if exp == exp_name]
    if not selected:
        continue

    finals = np.array([p["cil_snapshots"][-1] for _, p in selected], dtype=float)
    mean = finals.mean(axis=0)
    std = finals.std(axis=0, ddof=1) if len(finals) > 1 else np.zeros_like(mean)

    x = np.arange(1, len(mean)+1)
    plt.figure(figsize=(8, 4))
    plt.errorbar(x, mean, yerr=std, fmt="o-", capsize=3)
    plt.xlabel("Task")
    plt.ylabel("Final CIL accuracy (%)")
    plt.title(exp_name)
    plt.tight_layout()
    plt.savefig(OUT/f"{exp_name}_final_task_accuracy.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# Step 54 — Learning/forgetting trajectory heatmaps (mean across seeds)

for exp_name in ACTIVE_EXPERIMENTS:
    selected = [p["cil_snapshots"] for exp, seed, p in snapshot_rows if exp == exp_name]
    if not selected:
        continue

    T = max(len(x) for x in selected)
    cube = np.full((len(selected), T, T), np.nan)

    for si, snapshots in enumerate(selected):
        for t, row in enumerate(snapshots):
            cube[si, t, :len(row)] = row

    mean_matrix = np.nanmean(cube, axis=0)

    plt.figure(figsize=(7, 6))
    im = plt.imshow(mean_matrix, aspect="auto")
    plt.colorbar(im, label="CIL accuracy (%)")
    plt.xlabel("Evaluated task")
    plt.ylabel("After training task")
    plt.title(f"{exp_name} — mean accuracy matrix")
    plt.tight_layout()
    plt.savefig(OUT/f"{exp_name}_accuracy_matrix.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# Step 55 — Generate LaTeX result rows

latex_rows = []
for _, r in agg.iterrows():
    latex_rows.append(
        f"{r['experiment']} & "
        f"{r['CIL_mean']:.2f} $\\pm$ {r['CIL_std']:.2f} & "
        f"{r['TIL_mean']:.2f} $\\pm$ {r['TIL_std']:.2f} & "
        f"{r['FF_mean']:.2f} $\\pm$ {r['FF_std']:.2f} \\\\"
    )

latex_text = "\n".join(latex_rows)
print(latex_text)
(OUT/"latex_result_rows.txt").write_text(latex_text)


In [ ]:
# Step 56 — Save environment versions

import pkgutil

env_info = {
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "repo_commit": commit,
}
(OUT/"environment.json").write_text(json.dumps(env_info, indent=2))
print(json.dumps(env_info, indent=2))


In [ ]:
# Step 57 — Collect checkpoint paths

checkpoints = sorted(REPO.glob("experiments/**/*.pth"))
print("Checkpoints found:", len(checkpoints))
for p in checkpoints[:50]:
    print(p)


In [ ]:
# Step 58 — Copy result-relevant checkpoints if desired
# Disabled by default to avoid duplicating large files.
COPY_CHECKPOINTS_TO_OUTPUT = False

if COPY_CHECKPOINTS_TO_OUTPUT:
    ckpt_dir = OUT/"checkpoints"
    ckpt_dir.mkdir(exist_ok=True)
    for p in checkpoints:
        shutil.copy2(p, ckpt_dir/p.name)
    print("Copied:", len(checkpoints))
else:
    print("Checkpoint copying disabled; originals remain in the repository experiment directory.")


In [ ]:
# Step 59 — Validate expected run count

expected = len(ACTIVE_EXPERIMENTS) * len(SEEDS)
actual = len(results)

print("Expected runs:", expected)
print("Completed runs:", actual)

if actual != expected:
    print("WARNING: not all runs completed. Re-run Cell 41; completed jobs will be skipped.")
else:
    print("All runs completed.")


In [ ]:
# Step 60 — Validate finite metrics

metric_cols = ["CIL_FAA", "TIL_FAA", "Final_Forgetting"]
finite_ok = np.isfinite(results[metric_cols].to_numpy()).all()
print("All metrics finite:", finite_ok)
if not finite_ok:
    raise RuntimeError("Non-finite metric detected.")


In [ ]:
# Step 61 — Save a compact final report

report_lines = [
    "IDER ICLR 2026 — Kaggle reproduction",
    f"Repository commit: {commit}",
    f"Completed runs: {len(results)}/{len(ACTIVE_EXPERIMENTS)*len(SEEDS)}",
    "Paper comparison: Table 1 FAA + Table 7 FF",
    "TIL FAA: extra diagnostic only (not a main paper-table metric)",
    "",
]

for _, r in agg.iterrows():
    report_lines += [
        r["experiment"],
        f"  CIL FAA: {r['CIL_mean']:.2f} ± {r['CIL_std']:.2f}",
        f"  FF:      {r['FF_mean']:.2f} ± {r['FF_std']:.2f}",
        f"  TIL FAA (extra): {r['TIL_mean']:.2f} ± {r['TIL_std']:.2f}",
        "",
    ]

report = "\n".join(report_lines)
print(report)
(OUT/"FINAL_REPORT.txt").write_text(report)


In [ ]:
# Step 62 — Package all lightweight outputs

archive = shutil.make_archive(
    str(ROOT/"IDER_ICLR2026_Kaggle_Results"),
    "zip",
    root_dir=OUT
)
print("Archive:", archive)
